Assemble a reference ``df_parts`` from per-part window sets, so each sequence part can use its **own** generator. Here a biologically-motivated background is built with a coil-propensity JMD-N, an alpha-helix TMD, and a coil-propensity JMD-C - each from a separate ``AAWindowSampler.sample_synthetic`` call:

In [ ]:
import pandas as pd
import aaanalysis as aa

df_seq = aa.load_dataset(name="DOM_GSEC", n=8)
sf = aa.SequenceFeature()
df_parts = sf.get_df_parts(df_seq=df_seq, list_parts=["jmd_n", "tmd", "jmd_c"])
n_ref = len(df_parts)

aws = aa.AAWindowSampler(random_state=0)
dict_parts = {
    "jmd_n": aws.sample_synthetic(df_seq=df_seq, n=n_ref, window_size=10, generator="coil"),
    "tmd":   aws.sample_synthetic(df_seq=df_seq, n=n_ref, window_size=20, generator="alpha_helix"),
    "jmd_c": aws.sample_synthetic(df_seq=df_seq, n=n_ref, window_size=10, generator="coil"),
}
df_parts_ref = aa.SequenceFeature.get_df_parts_from_windows(dict_parts)
df_parts_ref.head()

Concatenate the per-part reference with the real parts as the reference class (0) and run CPP. A smaller ``split_kws`` is used because the base JMD parts are short:

In [ ]:
df_all = pd.concat([df_parts, df_parts_ref])
labels = [1] * len(df_parts) + [0] * len(df_parts_ref)
split_kws = sf.get_split_kws(n_split_max=5, steps_pattern=[3, 4], n_min=2, n_max=3, len_max=8)
df_feat = aa.CPP(df_parts=df_all, split_kws=split_kws).run(labels=labels, n_filter=5)
df_feat[["feature", "abs_auc"]].head()